# Diabetes Risk Explorer
## Part 2 — Modelling and evaluation

**Question:** Can anonymous survey features classify self-reported prediabetes or diabetes?

This notebook is for education and portfolio work only. It is not a clinical model and must not be used for diagnosis, advice, or decisions about people.


### What this section demonstrates
- Why accuracy is not enough for imbalanced data.
- How to create a fair train/test evaluation setup.
- How a simple, explainable baseline compares with a nonlinear model.
- How to state limitations honestly.


In [ ]:
!pip -q install ucimlrepo scikit-learn seaborn


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, ConfusionMatrixDisplay

sns.set_theme(style="whitegrid")


In [ ]:
# Load the same official UCI dataset used in Part 1
dataset = fetch_ucirepo(id=891)
X = dataset.data.features.copy()
y = dataset.data.targets["Diabetes_binary"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print(f"Training rows: {len(X_train):,}")
print(f"Held-out test rows: {len(X_test):,}")
print(f"Positive-class rate: {y.mean():.1%}")


### Two deliberately different models
**Logistic regression** is a transparent baseline. `class_weight="balanced"` asks it to pay more attention to the rarer positive class.

**Histogram gradient boosting** can learn nonlinear patterns and interactions. It may rank respondents well but still make poor yes/no decisions at a default threshold of 0.50.


In [ ]:
models = {
    "Logistic regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))
    ]),
    "Histogram gradient boosting": HistGradientBoostingClassifier(
        max_iter=250, learning_rate=0.08, max_leaf_nodes=31,
        l2_regularization=1.0, random_state=42
    )
}

results = {}
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    probability = model.predict_proba(X_test)[:, 1]
    predicted_class = (probability >= 0.50).astype(int)
    predictions[name] = (probability, predicted_class)
    results[name] = {
        "accuracy": accuracy_score(y_test, predicted_class),
        "precision": precision_score(y_test, predicted_class),
        "recall": recall_score(y_test, predicted_class),
        "f1": f1_score(y_test, predicted_class),
        "roc_auc": roc_auc_score(y_test, probability),
        "average_precision": average_precision_score(y_test, probability)
    }

metrics = pd.DataFrame(results).T.round(3)
metrics


In [ ]:
# Compare the metrics visually
ax = metrics.plot(kind="bar", figsize=(11, 5))
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Held-out test metrics")
ax.legend(ncol=3, loc="lower center", bbox_to_anchor=(0.5, -0.35))
plt.show()


In [ ]:
# Inspect one classification threshold for the nonlinear model
probability, predicted_class = predictions["Histogram gradient boosting"]
ConfusionMatrixDisplay.from_predictions(y_test, predicted_class, cmap="YlGnBu", colorbar=False)
plt.title("Gradient boosting at threshold = 0.50")
plt.show()


### How to discuss these results
The nonlinear model can rank examples more strongly than the baseline (see ROC-AUC and average precision), but a 0.50 threshold misses many reported cases. This does **not** mean the logistic model is universally better. It shows why a real model needs a threshold selected according to the consequences of false positives and false negatives—something this educational project cannot decide for healthcare.

**Never claim the model diagnoses diabetes.** It classifies a self-reported survey label in one historical dataset.


In [ ]:
# Optional next step: compare recall and precision at different thresholds
thresholds = [0.10, 0.20, 0.30, 0.40, 0.50]
tradeoff = []
for threshold in thresholds:
    prediction = (probability >= threshold).astype(int)
    tradeoff.append({
        "threshold": threshold,
        "precision": precision_score(y_test, prediction),
        "recall": recall_score(y_test, prediction),
        "f1": f1_score(y_test, prediction)
    })
pd.DataFrame(tradeoff).round(3)
